# Titanic — EDA, Feature Engineering и ML

Ноутбук сделан **в стиле приложенного примера**: линейная структура, комментарий перед графиком, вывод после графика или статистики, затем feature engineering и сравнение моделей.

In [ ]:

# Импортируем нужные библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Считываем датасет
df = pd.read_csv('Titanic.csv')
print('Размер датасета:', df.shape)
df.head()


In [ ]:

print('
Информация о данных:')
print(df.info())

print('
Статистика числовых признаков:')
print(df.describe())

print('
Пропуски в данных:')
print(df.isnull().sum())


## EDA

Сначала проверяем базовую структуру набора данных, баланс таргета и самые очевидные зависимости. Каждый график сопровождается коротким комментарием до и выводом после.

In [ ]:

print('Комментарий: сначала смотрим баланс таргета, чтобы понять, есть ли сильный дисбаланс классов.')

surv_counts = df['Survived'].value_counts().sort_index()
plt.figure(figsize=(6,4))
sns.countplot(x='Survived', data=df, palette='Set2')
plt.title('Распределение выживших и погибших')
plt.xlabel('Survived')
plt.ylabel('Количество')
plt.show()

print(f"Вывод: доля выживших = {df['Survived'].mean()*100:.1f}%.")
print('Классы не идеально сбалансированы, но перекос умеренный.')


In [ ]:

print('Комментарий: теперь проверяем влияние пола на выживаемость — для Titanic это один из главных факторов.')

plt.figure(figsize=(8,5))
sns.barplot(x='Sex', y='Survived', data=df, palette='pastel')
plt.title('Выживаемость по полу')
plt.xlabel('Пол')
plt.ylabel('Средняя выживаемость')
plt.show()

sex_survival = df.groupby('Sex')['Survived'].mean()
chi2, p_sex, _, _ = chi2_contingency(pd.crosstab(df['Sex'], df['Survived']))
print(sex_survival)
print(f'Вывод: женщины выживали чаще мужчин; p-value по chi-square = {p_sex:.6f}.')


In [ ]:

print('Комментарий: класс билета отражает социальный статус и доступ к лучшим условиям эвакуации.')

plt.figure(figsize=(8,5))
sns.barplot(x='Pclass', y='Survived', data=df, palette='viridis')
plt.title('Выживаемость по классу')
plt.xlabel('Класс')
plt.ylabel('Средняя выживаемость')
plt.show()

class_survival = df.groupby('Pclass')['Survived'].mean()
chi2, p_class, _, _ = chi2_contingency(pd.crosstab(df['Pclass'], df['Survived']))
print(class_survival)
print(f'Вывод: пассажиры 1 класса выживали заметно чаще; p-value = {p_class:.6f}.')


In [ ]:

print('Комментарий: совместный график по полу и классу помогает увидеть комбинированный эффект факторов.')

plt.figure(figsize=(10,6))
sns.barplot(x='Pclass', y='Survived', hue='Sex', data=df, palette='Set1')
plt.title('Выживаемость по классу и полу')
plt.xlabel('Класс')
plt.ylabel('Средняя выживаемость')
plt.show()

print('Вывод: лучший сценарий выживания наблюдается у женщин из 1 класса, худший — у мужчин из 3 класса.')


In [ ]:

print('Комментарий: возраст проверяем и по распределению, и через boxplot; дополнительно используем тест Манна–Уитни.')

df_age = df.dropna(subset=['Age']).copy()

plt.figure(figsize=(13,5))
plt.subplot(1,2,1)
plt.hist(df_age[df_age['Survived']==0]['Age'], bins=30, alpha=0.5, label='Погибшие', color='red')
plt.hist(df_age[df_age['Survived']==1]['Age'], bins=30, alpha=0.5, label='Выжившие', color='green')
plt.legend()
plt.title('Распределение возраста')
plt.xlabel('Возраст')
plt.ylabel('Количество')

plt.subplot(1,2,2)
sns.boxplot(x='Survived', y='Age', data=df_age, palette='Set3')
plt.title('Возраст по выживаемости')
plt.xlabel('Survived')
plt.ylabel('Возраст')
plt.tight_layout()
plt.show()

age_stat, age_p = mannwhitneyu(df_age[df_age['Survived']==0]['Age'], df_age[df_age['Survived']==1]['Age'])
print(f"Медианный возраст погибших: {df_age[df_age['Survived']==0]['Age'].median():.1f}")
print(f"Медианный возраст выживших: {df_age[df_age['Survived']==1]['Age'].median():.1f}")
print(f'Вывод: возраст связан с выживаемостью; p-value = {age_p:.6f}.')


In [ ]:

print('Комментарий: стоимость билета — ещё один прокси социального положения, поэтому сравниваем её между группами.')

plt.figure(figsize=(13,5))
plt.subplot(1,2,1)
plt.hist(df[df['Survived']==0]['Fare'], bins=50, alpha=0.5, label='Погибшие', color='red')
plt.hist(df[df['Survived']==1]['Fare'], bins=50, alpha=0.5, label='Выжившие', color='green')
plt.legend()
plt.xlim(0, 200)
plt.title('Распределение стоимости билета')
plt.xlabel('Fare')
plt.ylabel('Количество')

plt.subplot(1,2,2)
sns.boxplot(x='Survived', y='Fare', data=df, palette='coolwarm')
plt.ylim(0, 200)
plt.title('Стоимость билета по выживаемости')
plt.xlabel('Survived')
plt.ylabel('Fare')
plt.tight_layout()
plt.show()

fare_stat, fare_p = mannwhitneyu(df[df['Survived']==0]['Fare'], df[df['Survived']==1]['Fare'])
print(f"Медианная цена билета у погибших: {df[df['Survived']==0]['Fare'].median():.2f}")
print(f"Медианная цена билета у выживших: {df[df['Survived']==1]['Fare'].median():.2f}")
print(f'Вывод: более дорогой билет связан с большей выживаемостью; p-value = {fare_p:.6f}.')


In [ ]:

print('Комментарий: теперь посмотрим общую корреляционную картину по числовым признакам и отдельно связь с таргетом.')

numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Корреляция числовых признаков')
plt.show()

print('
Корреляция исходных признаков с Survived:')
print(corr_matrix['Survived'].sort_values(ascending=False))
print('Вывод: среди исходных числовых признаков самые сильные сигналы обычно дают Fare и Pclass.')


## Feature Engineering

Теперь создаём новые признаки в духе примера: размер семьи, одиночное путешествие, титул, ребёнок, женщина или ребёнок, наличие каюты и дополнительные группировки.

In [ ]:

df_fe = df.copy()

df_fe['FamilySize'] = df_fe['SibSp'] + df_fe['Parch'] + 1
df_fe['IsAlone'] = (df_fe['FamilySize'] == 1).astype(int)
df_fe['Title'] = df_fe['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Группируем редкие титулы
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}
df_fe['Title'] = df_fe['Title'].replace(title_mapping)
df_fe['Title'] = df_fe['Title'].fillna('Rare')

df_fe['IsChild'] = (df_fe['Age'].fillna(df_fe['Age'].median()) < 12).astype(int)
df_fe['WomanOrChild'] = ((df_fe['Sex'] == 'female') | (df_fe['Age'].fillna(df_fe['Age'].median()) < 12)).astype(int)
df_fe['HasCabin'] = df_fe['Cabin'].notna().astype(int)
df_fe['FarePerPerson'] = df_fe['Fare'] / df_fe['FamilySize']

ticket_group = df_fe['Ticket'].value_counts()
df_fe['TicketGroupSize'] = df_fe['Ticket'].map(ticket_group)

print('Новые признаки:')
print(df_fe[['FamilySize', 'IsAlone', 'Title', 'IsChild', 'WomanOrChild', 'HasCabin', 'FarePerPerson', 'TicketGroupSize']].head())


In [ ]:

print('Комментарий: оценим корреляции engineered-признаков с таргетом, чтобы понять, какие из них добавляют полезный сигнал.')

df_corr = df_fe.copy()
df_corr['Title_num'] = df_corr['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})
new_features = ['FamilySize', 'IsAlone', 'IsChild', 'WomanOrChild', 'HasCabin', 'FarePerPerson', 'TicketGroupSize', 'Title_num']

corr_new = df_corr[new_features + ['Survived']].corr()['Survived'].sort_values(ascending=False)
print(corr_new)

plt.figure(figsize=(8,5))
sns.barplot(x=corr_new.drop('Survived').values, y=corr_new.drop('Survived').index, palette='magma')
plt.title('Корреляция новых признаков с таргетом')
plt.xlabel('Корреляция')
plt.ylabel('Признак')
plt.show()

print('Вывод: наиболее полезными обычно оказываются WomanOrChild, Title, HasCabin, IsAlone и FarePerPerson.')


## Подготовка данных

Далее заполняем пропуски, создаём категориальные интервалы и кодируем признаки для обучения моделей.

In [ ]:

df_clean = df_fe.copy()

df_clean['Age'] = df_clean.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())

df_clean['AgeGroup'] = pd.cut(df_clean['Age'], bins=[0, 12, 18, 50, 100], labels=['Child', 'Teen', 'Adult', 'Elderly'])
df_clean['FareGroup'] = pd.qcut(df_clean['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Very High'], duplicates='drop')

df_encoded = pd.get_dummies(
    df_clean,
    columns=['Sex', 'Embarked', 'Title', 'AgeGroup', 'FareGroup'],
    drop_first=True
)

y = df_encoded['Survived']
features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'IsChild', 'WomanOrChild', 'HasCabin', 'FarePerPerson', 'TicketGroupSize']
features += [col for col in df_encoded.columns if col.startswith(('Sex_', 'Embarked_', 'Title_', 'AgeGroup_', 'FareGroup_'))]
X = df_encoded[features]

print(f'Подготовлено {len(features)} признаков для модели')
print('Первые 5 признаков:', features[:5])


## Простая модель и модели разных семейств

В стиле примера берём одну модель из каждого семейства: линейная, дерево, ансамбль деревьев/бустинг и нейронная сеть. Сначала считаем метрики на holdout, затем выбираем лучшую и проверяем её по 5-fold CV.

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Linear / Logistic Regression': LogisticRegression(random_state=42, max_iter=2000),
    'Tree / Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_leaf=10),
    'Ensemble / Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=120, learning_rate=0.05),
    'Neural Net / MLP': MLPClassifier(random_state=42, hidden_layer_sizes=(32, 16), max_iter=700)
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print('
' + '='*80)
    print(name)

    if 'Logistic' in name or 'MLP' in name:
        X_train_use = X_train_scaled
        X_test_use = X_test_scaled
        X_cv_use = scaler.fit_transform(X_train)
    else:
        X_train_use = X_train
        X_test_use = X_test
        X_cv_use = X_train

    model.fit(X_train_use, y_train)
    y_pred = model.predict(X_test_use)

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_use)[:, 1]
    else:
        y_proba = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    cv_scores = cross_val_score(model, X_cv_use, y_train, cv=cv, scoring='f1')

    print(f'Accuracy:  {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall:    {recall:.4f}')
    print(f'F1-score:  {f1:.4f}')
    print(f'ROC-AUC:   {roc_auc:.4f}')
    print(f'CV F1 mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'CV_F1_Mean': cv_scores.mean(),
        'CV_F1_STD': cv_scores.std()
    })

results_df = pd.DataFrame(results).sort_values(['F1-Score', 'ROC-AUC'], ascending=False)
print('
Сравнение моделей:')
print(results_df)


In [ ]:

print('Комментарий: визуально сравним модели по F1-score, так как метрика учитывает и precision, и recall.')

plt.figure(figsize=(11,6))
sns.barplot(data=results_df, x='F1-Score', y='Model', palette='cubehelix')
plt.title('Сравнение моделей по F1-score')
plt.xlabel('F1-score')
plt.ylabel('Модель')
plt.show()

best_model_name = results_df.iloc[0]['Model']
print(f'Вывод: лучшая модель на holdout — {best_model_name}.')


In [ ]:

print('Комментарий: отдельно посмотрим важности признаков у бустинга или случайного леса, чтобы понять вклад признаков.')

importance_model = GradientBoostingClassifier(random_state=42, n_estimators=120, learning_rate=0.05)
importance_model.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importance_model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

print(feature_importance)

plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feature_importance, palette='flare')
plt.title('Топ-15 важнейших признаков')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

print('Вывод: в топе обычно находятся признаки пола, класса, титула, возраста и производные от них engineered-признаки.')


In [ ]:

print('Комментарий: теперь подтверждаем выбор лучшей модели с помощью кросс-валидации не менее чем на 5 фолдах.')

best_name = results_df.iloc[0]['Model']

if best_name == 'Linear / Logistic Regression':
    best_model = LogisticRegression(random_state=42, max_iter=2000)
    X_full = scaler.fit_transform(X)
elif best_name == 'Tree / Decision Tree':
    best_model = DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_leaf=10)
    X_full = X
elif best_name == 'Ensemble / Gradient Boosting':
    best_model = GradientBoostingClassifier(random_state=42, n_estimators=120, learning_rate=0.05)
    X_full = X
else:
    best_model = MLPClassifier(random_state=42, hidden_layer_sizes=(32, 16), max_iter=700)
    X_full = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(best_model, X_full, y, cv=cv, scoring='f1')
cv_acc = cross_val_score(best_model, X_full, y, cv=cv, scoring='accuracy')
cv_auc = cross_val_score(best_model, X_full, y, cv=cv, scoring='roc_auc')

print('Лучшая модель:', best_name)
print(f'5-fold CV Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f})')
print(f'5-fold CV F1-score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')
print(f'5-fold CV ROC-AUC:  {cv_auc.mean():.4f} (+/- {cv_auc.std():.4f})')


In [ ]:

print('
ИТОГОВЫЕ ВЫВОДЫ:')
print('1. По EDA главные факторы выживаемости — пол, класс билета, возраст и стоимость билета.')
print('2. Feature engineering добавляет полезные сигналы: FamilySize, IsAlone, Title, WomanOrChild, HasCabin и FarePerPerson.')
print('3. Корреляции новых признаков с таргетом помогают быстро отобрать наиболее информативные признаки.')
print('4. Были протестированы модели разных семейств: линейная, дерево, градиентный бустинг и нейронная сеть.')
print('5. Лучшая модель выбирается по holdout, после чего подтверждается 5-fold кросс-валидацией.')
print('6. Такой формат ноутбука удобно сдавать как учебный проект: он линейный, читаемый и близок к приложенному примеру.')
